# Piero YOLOv8s Barcode Detector

Run the same detector as `run.py`, then inspect the JSON and debug images.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from IPython.display import Image, display

def find_project_dir():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / "run.py").exists() and base.name == "piero_yolov8s":
            return base
        candidate = base / "piero_yolov8s"
        if (candidate / "run.py").exists():
            return candidate
    raise RuntimeError("Could not find piero_yolov8s/run.py")

PROJECT = find_project_dir()
REPO = PROJECT.parent
PYTHON = REPO / "venv" / "bin" / "python"
PYTHON = PYTHON if PYTHON.exists() else Path(sys.executable)

print("project:", PROJECT)
print("python:", PYTHON)

## Run Detection

Change `PAGES`, `CONF`, or `IMGSZ` here, then run the cell.

In [ ]:
PAGES = "1-21"
DPI = 250
IMGSZ = 1280
CONF = 0.10
OUT = PROJECT / "output-notebook-all-pages"
SHOW_IMAGES = False
MAX_DEBUG_IMAGES = 6

cmd = [
    str(PYTHON),
    str(PROJECT / "run.py"),
    "--pages", PAGES,
    "--dpi", str(DPI),
    "--imgsz", str(IMGSZ),
    "--conf", str(CONF),
    "--out", str(OUT),
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Detection Summary

In [ ]:
detections_path = OUT / "detections.json"
data = json.loads(detections_path.read_text())

print("detections:", detections_path)
print("conf:", data["conf"], "imgsz:", data["imgsz"], "elapsed:", data["elapsed_seconds"])
for page in data["pages"]:
    counts = {}
    for det in page["detections"]:
        counts[det["class_name"]] = counts.get(det["class_name"], 0) + 1
    print(f"page {page['page']}: {len(page['detections'])} detections {counts}")

## Debug Images

In [ ]:
debug_paths = [OUT / page["debug"] for page in data["pages"]]
print("debug images:")
for path in debug_paths:
    print(path)

if SHOW_IMAGES:
    for page in data["pages"][:MAX_DEBUG_IMAGES]:
        print(f"page {page['page']}")
        display(Image(filename=str(OUT / page["debug"])))
else:
    print(f"Set SHOW_IMAGES = True to preview the first {MAX_DEBUG_IMAGES} debug images in the notebook.")
